In [4]:
!pip install groq python-dotenv

In [6]:
from os import environ
from google.colab import userdata

environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

print("Groq client initialized successfully!")

Groq client initialized successfully!


In [17]:
# MODEL_NAME = "mixtral-8x7b-32768"
MODEL_NAME = "llama-3.1-8b-instant"

MODEL_CONFIG = {
    "technical": {
        "system_prompt": """
You are a Technical Support Expert.
Be precise, rigorous, and code-focused.
Provide debugging steps and corrected code if needed.
Avoid unnecessary explanations.
"""
    },
    "billing": {
        "system_prompt": """
You are a Billing Support Expert.
Be empathetic, professional, and policy-driven.
Explain refund policies clearly and suggest next steps.
"""
    },
    "general": {
        "system_prompt": """
You are a friendly Customer Support Assistant.
Handle general queries and casual conversation professionally.
"""
    }
}

In [18]:
def route_prompt(user_input):
    routing_prompt = f"""
Classify the following user query into one of these categories:
[technical, billing, general, tool]

Return ONLY the category name.

User Query:
{user_input}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        messages=[
            {"role": "system", "content": "You are an intent classification engine."},
            {"role": "user", "content": routing_prompt}
        ]
    )

    category = response.choices[0].message.content.strip().lower()
    return category

In [19]:
def fetch_bitcoin_price():
    return "The current price of Bitcoin is approximately $62,500 (mock value)."

In [20]:
def process_request(user_input):

    category = route_prompt(user_input)
    print(f"[Router Decision]: {category}")

    if category == "tool":
        return fetch_bitcoin_price()

    if category not in MODEL_CONFIG:
        category = "general"

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content

In [21]:
print(process_request("My python script is throwing an IndexError on line 5."))
print(process_request("I was charged twice for my subscription this month."))
print(process_request("Hi there! What services do you offer?"))
print(process_request("What is the current price of Bitcoin?"))

[Router Decision]: technical
To troubleshoot the issue, I'll need more information. Can you please provide:

1. The code (at least line 5 and surrounding lines)
2. The full error message (including the IndexError message)
3. The input or data you're using when running the script

Please paste the relevant code, and I'll help you identify the cause of the IndexError and suggest corrections.
[Router Decision]: billing
I'm so sorry to hear that you were charged twice for your subscription. I'm here to help you resolve this issue as quickly as possible.

Our refund policy states that if a duplicate charge occurs, we will refund the excess amount within 5-7 business days after we are notified. Since this is an error on our part, we are happy to refund the duplicate charge.

To confirm, I would like to verify some information to ensure that we process the refund accurately. Can you please confirm the following:

1. Your subscription account details (email address or username associated with 